# 从零实现 ReAct，并对比 CoT / Self-Consistency / ReAct

**任务**：GSM8K 风格数学题。我们手动构造 10 道带噪声的题目（部分需要查 *维基风格事实*），用三种范式对比：

1. **Zero-shot CoT**：直接 `Let's think step by step`。
2. **Self-Consistency**：CoT 采样 5 次取多数。
3. **ReAct**：自然语言 prompt 触发 `Thought / Action / Observation` 循环，提供 `calculator` 与 `lookup` 两个工具。

目的：直观感受三者的准确率与 token 开销。

> 注：本 notebook 不连真正的 GSM8K（避免依赖），用一个 mini 自制集说明问题。要换成真实 GSM8K，把 `MINI_DATASET` 改成 `datasets.load_dataset('gsm8k', 'main', split='test[:50]')` 即可。

In [ ]:
import os, sys, re, json, math, statistics
sys.path.append(os.path.abspath('../..'))
from utils.llm_client import LLMClient
from utils.eval_utils import normalize_answer, RunResult, compare_runs

client = LLMClient(temperature=0.0)
client.model

## 1. Mini 数据集

10 道题，部分需要外部知识（用 `lookup` 工具拿）。

In [ ]:
MINI_DATASET = [
    {'q': '一个篮子有 12 个苹果，吃了 5 个，又买了 8 个，现在有几个？', 'a': '15'},
    {'q': '一辆车每小时 60 公里，行驶 2.5 小时，走了多少公里？', 'a': '150'},
    {'q': '一个班 30 人，其中 60% 是女生，女生有几人？', 'a': '18'},
    {'q': '小明买了 3 支笔，每支 4 元，付了 20 元，找零多少元？', 'a': '8'},
    {'q': '一根绳子长 12 米，剪成 3 段，每段多长？', 'a': '4'},
    {'q': '光速约为多少 km/s？请只回答整数。', 'a': '300000'},  # 需 lookup
    {'q': '北京到上海高铁约 1300 公里，时速 350，需要几小时？保留 2 位小数。', 'a': '3.71'},
    {'q': '半径 7 的圆面积，pi 取 3.14。', 'a': '153.86'},
    {'q': '5 的阶乘是多少？', 'a': '120'},
    {'q': '一杯水 250ml，喝了 2/5，还剩多少 ml？', 'a': '150'},
]
len(MINI_DATASET)

## 2. Zero-shot CoT

In [ ]:
COT_PROMPT = (
    "请一步步思考下面这道题，并在最后一行只输出最终数值答案（不要带单位、不要解释）。\n\n"
    "题目：{q}\n"
)

def run_cot(q: str) -> str:
    out = client.chat([{'role': 'user', 'content': COT_PROMPT.format(q=q)}])
    return out['text']

def eval_run(name: str, runner) -> RunResult:
    correct = 0
    before_in = client.usage.input_tokens
    before_out = client.usage.output_tokens
    for item in MINI_DATASET:
        pred = normalize_answer(runner(item['q']))
        if pred == normalize_answer(item['a']):
            correct += 1
    return RunResult(
        name=name,
        correct=correct,
        total=len(MINI_DATASET),
        tokens_in=client.usage.input_tokens - before_in,
        tokens_out=client.usage.output_tokens - before_out,
    )

r_cot = eval_run('zero-shot CoT', run_cot)
print(r_cot)

## 3. Self-Consistency（CoT × 5 + 多数投票）

In [ ]:
from collections import Counter

sc_client = LLMClient(temperature=0.7)  # 高温采样

def run_sc(q: str, k: int = 5) -> str:
    answers = []
    for _ in range(k):
        out = sc_client.chat([{'role': 'user', 'content': COT_PROMPT.format(q=q)}])
        answers.append(normalize_answer(out['text']))
    return Counter(answers).most_common(1)[0][0]

before_in = sc_client.usage.input_tokens
before_out = sc_client.usage.output_tokens
correct = 0
for item in MINI_DATASET:
    if run_sc(item['q']) == normalize_answer(item['a']):
        correct += 1
r_sc = RunResult(
    name='self-consistency k=5',
    correct=correct,
    total=len(MINI_DATASET),
    tokens_in=sc_client.usage.input_tokens - before_in,
    tokens_out=sc_client.usage.output_tokens - before_out,
)
print(r_sc)

## 4. 手撸 ReAct

我们解析自然语言里的 `Action: tool[arg]`，**不**用 Anthropic 原生 tool use（这样能更直接看到 ReAct 的形态）。

In [ ]:
REACT_PROMPT = (
    "你是一个解题 Agent。请按如下格式交替输出。\n"
    "可用工具：\n"
    "  - calculator[expr]：计算 Python 表达式，例如 calculator[12-5+8]\n"
    "  - lookup[query]：查事实常识，例如 lookup[光速 km/s]\n"
    "  - finish[answer]：给出最终数值答案，answer 不带单位\n"
    "\n"
    "严格按以下格式逐行输出，每轮只输出一个 Thought 和一个 Action：\n"
    "Thought 1: ...\nAction 1: tool[arg]\n（系统会给 Observation）\n\n"
    "题目：{q}\n"
)

_LOOKUP_KB = {
    '光速': '光速约为 299792 km/s，常被近似为 300000 km/s',
    'pi': 'pi ≈ 3.14159',
}

def tool_calculator(expr: str) -> str:
    try:
        return str(eval(expr, {'__builtins__': {}}, {'math': math}))
    except Exception as e:
        return f'Error: {e}'

def tool_lookup(query: str) -> str:
    for k, v in _LOOKUP_KB.items():
        if k in query:
            return v
    return '未找到。请基于自有知识作答。'

ACTION_RE = re.compile(r'Action\s*\d*:\s*(\w+)\[(.*?)\]')

def run_react(q: str, max_steps: int = 6) -> str:
    history = REACT_PROMPT.format(q=q)
    for _ in range(max_steps):
        out = client.chat([{'role': 'user', 'content': history}])['text']
        history += out + '\n'
        m = ACTION_RE.search(out)
        if not m:
            return out
        tool, arg = m.group(1), m.group(2)
        if tool == 'finish':
            return arg
        if tool == 'calculator':
            obs = tool_calculator(arg)
        elif tool == 'lookup':
            obs = tool_lookup(arg)
        else:
            obs = f'未知工具 {tool}'
        history += f'Observation: {obs}\n'
    return '[max_steps]'

r_react = eval_run('ReAct', run_react)
print(r_react)

## 5. 横向对比

In [ ]:
print(compare_runs(r_cot, r_sc, r_react))

## 6. 观察与思考

- ReAct 在「需要外部知识」（如光速）的题目上明显更可靠。
- Self-Consistency 在「LLM 内部偶发错误」的题目上能拉高准确率，但 token 开销 ≈ 5×。
- 真实生产里更常见的搭配是 *ReAct + tool use*（结构化 JSON）+ 关键节点 self-consistency 投票。

## 进阶练习

1. 把数据集换成真正的 GSM8K 100 题，画 acc-token 散点图。
2. 加一层 **Reflexion**：失败题目让 LLM 写反思，下一轮拼到 prompt 里再跑一遍。
3. 把 `ReAct` 用 Anthropic 原生 tool use 改写一遍，比较 *prompt 长度* 与 *错误率*。